# 01 - Attention by Hand

Compute attention scores, normalized weights, and one context vector without trainable projections.

This is a simplified, unscaled form of self-attention. Later notebooks introduce trainable query, key, and value projections.

Here, we calculate raw attention scores using the dot product between each input vector and the query. A larger score means that input receives more attention from this query in this simplified example.

In [5]:
import torch

inputs = torch.tensor([
    [0.43, 0.15, 0.89],
    [0.55, 0.87, 0.66],
    [0.57, 0.85, 0.64],
    [0.22, 0.58, 0.33],
    [0.77, 0.25, 0.10],
    [0.05, 0.80, 0.55],
])

query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


Next, we normalize the scores so they sum to 1. This simple approach works for these positive scores, but it is not the general attention normalization method. In practice, attention uses softmax.

In [6]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


Softmax is used because it converts scores into positive weights that sum to 1, while remaining differentiable during training.

Softmax produces positive attention weights that sum to 1. The resulting context vector is a weighted combination of the input vectors. Below is a naive implementation.

In [7]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


This naive implementation can be numerically unstable because `exp` may overflow for large positive values or underflow for very negative values. `torch.softmax` uses a numerically stable implementation.

In [8]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights", attn_weights_2)
print("Sum: ", attn_weights_2.sum())

Attention weights tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:  tensor(1.)


Finally, we calculate one context vector for the selected query by multiplying each input vector by its corresponding attention weight and summing the results.

In [9]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate (inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])
